In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------ CONFIG ------------------

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

#benchmarks = [
#    "private_enterprise",
#    "social_media_cloud"
#]

loads = range(1, 10)

input_dir = "/home/hsd/workspace/trafpy/examples/comparison_generator/final_data"

# Existing ns3 outputs
output_dir = "/home/hsd/workspace/ns3-load-balance/results_new_test_1_sym"

# RTT outputs (change this)
#rtt_dir = "/home/hsd/workspace/ns3-load-balance/results_rtt_new_test"

new_rtt_dir = "/home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym"

save_dir = "/home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results"
Path(save_dir).mkdir(exist_ok=True)

ip_pattern = r'^\d+\.\d+\.\d+\.\d+$'

summary_results = {}

# ------------------ FUNCTIONS ------------------

def clean(df):
    return df[
        df['Src'].str.match(ip_pattern, na=False) &
        df['Dest'].str.match(ip_pattern, na=False)
    ].copy()


def ip_to_node(ip):
    last = int(ip.split('.')[-1])
    last1 = int(ip.split('.')[-2])

    leafcount = 2
    leaf = 0

    if last1 == 2:
        leaf = 1

    return ((last // 2) - 1 + (leaf * leafcount))


def add_time(df):

    df['start_time'] = (
        df['TimeFirstTxPacket']
        .astype(str)
        .str.replace('+', '', regex=False)
        .str.replace('ns', '', regex=False)
        .astype(float) / 1e9
    )

    return df


flow_columns = [
    "FlowID",
    "Src",
    "Dest",
    "TimeFirstRxPacket",
    "TimeFirstTxPacket",
    "TimeLastRxPacket",
    "TimeLastTxPacket",
    "FCT(s)",
    "TxPackets",
    "RxPackets",
    "LostPackets",
    "LossRate",
    "PDR",
    "LossPercent",
    "TxBytes",
    "RxBytes",
    "Throughput(Kbps)",
    "MeanDelay(ms)",
    "Jitter(ms)",
    "HopCount"
]


# ------------------ MATCH ------------------

def match_df(output_df, input_df, label):

    matches = []

    for _, in_row in input_df.iterrows():

        cand = output_df[
            (output_df.sn == in_row.sn) &
            (output_df.dn == in_row.dn)
        ].copy()

        if len(cand) == 0:
            continue

        cand['time_diff'] = abs(
            cand['start_time'] - in_row.event_time
        )

        best = cand.loc[cand['time_diff'].idxmin()]

        combined = {
            'flow_id': in_row.flow_id,
            'sn': in_row.sn,
            'dn': in_row.dn,
            'input_time': in_row.event_time,
            'flow_size': in_row.flow_size,

            f'time_diff_{label}': best['time_diff'],

            f'TxPackets_{label}': best['TxPackets'],
            f'RxPackets_{label}': best['RxPackets'],

            f'TxBytes_calc_{label}': best['TxPackets'] * 1400,
            f'RxBytes_calc_{label}': best['RxPackets'] * 1400,
        }

        for col in flow_columns:
            combined[f"{col}_{label}"] = best[col]

        matches.append(combined)

    return pd.DataFrame(matches)


# ------------------ MAIN ------------------

for benchmark in benchmarks:

    for load in loads:

        print(f"\n=== {benchmark} | Load 0.{load} ===")

        input_path = (
            f"{input_dir}/{benchmark}_load_{load}.csv"
        )

        try:

            input_df = pd.read_csv(input_path)

            timeout_values = ["true", "false"]

            algorithms = {
                "conga": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_Conga.csv",
                    nrows=12000
                ),

                #"conga_new": pd.read_csv(
                #    f"{output_dir}/{benchmark}_load_{load}_Conga_new.csv",
                #    nrows=12000
                #),

                #"conga-ecmp": pd.read_csv(
                #    f"{output_dir}/{benchmark}_load_{load}_Conga-ECMP_new.csv",
                #    nrows=12000
                #),

                "ecmp": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_ECMP.csv",
                    nrows=12000
                ),

                #"ecmp_new": pd.read_csv(
                #    f"{output_dir}/{benchmark}_load_{load}_ECMP_new.csv",
                #    nrows=12000
                #),

                #"rtt": pd.read_csv(
                #    f"{rtt_dir}/{benchmark}_load_{load}_RTT.csv",
                #    nrows=12000
                #),

                #"weighted": pd.read_csv(
                #    f"{new_rtt_dir}/{benchmark}_load_{load}_WEIGHTED_ECMP_RTT.csv",
                #    nrows=12000
                #),

                #"random": pd.read_csv(
                #    f"{new_rtt_dir}/{benchmark}_load_{load}_POWER_OF_2_RANDOM_RTT.csv",
                #    nrows=12000
                #),

                #"top2": pd.read_csv(
                #    f"{new_rtt_dir}/{benchmark}_load_{load}_POWER_OF_2_TOP2_RTT.csv",
                #    nrows=12000
                #)
            }

            for timeout in timeout_values:
            
                algorithms[f"weighted_{timeout}"] = pd.read_csv(
                    f"{new_rtt_dir}/{benchmark}_load_{load}_WEIGHTED_ECMP_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

                algorithms[f"random_{timeout}"] = pd.read_csv(
                    f"{new_rtt_dir}/{benchmark}_load_{load}_POWER_OF_2_RANDOM_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

                algorithms[f"lrtt_{timeout}"] = pd.read_csv(
                    f"{new_rtt_dir}/{benchmark}_load_{load}_LOWEST_RTT_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

                algorithms[f"top2_{timeout}"] = pd.read_csv(
                    f"{new_rtt_dir}/{benchmark}_load_{load}_POWER_OF_2_TOP2_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

        except Exception as e:
            print("Skipping:", e)
            continue

        # ------------------ CLEAN ------------------

        for label, df in algorithms.items():

            df = add_time(clean(df))

            df['sn'] = df['Src'].apply(ip_to_node)
            df['dn'] = df['Dest'].apply(ip_to_node)

            algorithms[label] = df

        # ------------------ MATCH ------------------

        matched = {}

        for label, df in algorithms.items():
            matched[label] = match_df(df, input_df, label)

        merge_keys = [
            'flow_id',
            'sn',
            'dn',
            'input_time',
            'flow_size'
        ]

        algorithms_list = list(algorithms.keys())

        final_df = matched[algorithms_list[0]]

        for algo in algorithms_list[1:]:
            final_df = pd.merge(
                final_df,
                matched[algo],
                on=merge_keys,
                how='inner'
            )

        if len(final_df) == 0:
            print("No matches.")
            continue

        # ------------------ DIFFERENCES ------------------

        reference = "ecmp"
        
        for algo in algorithms:
            if algo == reference:
                continue
        
            final_df[f"fct_diff_{reference}_{algo}"] = (
                final_df[f"FCT(s)_{reference}"]
                - final_df[f"FCT(s)_{algo}"]
            )

        # ------------------ SAVE CSV ------------------

        csv_path = (
            f"{save_dir}/{benchmark}_load_{load}_full.csv"
        )

        final_df.to_csv(csv_path, index=False)

        # ------------------ STATS ------------------

        stats = {}

        for algo in algorithms:
            fct = final_df[f"FCT(s)_{algo}"]

            stats[algo] = {
                "avg": fct.mean(),
                "p75": fct.quantile(0.75),
                "p90": fct.quantile(0.90),
                "p95": fct.quantile(0.95),
                "p99": fct.quantile(0.99),
            }

        
        for algo in stats:
            print(
                algo,
                "Avg:", stats[algo]["avg"],
                "P75:", stats[algo]["p75"],
                "P90:", stats[algo]["p90"],
                "P95:", stats[algo]["p95"],
                "P99:", stats[algo]["p99"],
            )

        if benchmark not in summary_results:

            summary_results[benchmark] = {
                "loads": []
            }

            for algo in algorithms:
                summary_results[benchmark][algo] = {
                "avg": [],
                "p75": [],
                "p90": [],
                "p95": [],
                "p99": []
                }

        summary_results[benchmark]["loads"].append(load / 10)

        for algo in algorithms:
            for metric in ["avg", "p75", "p90", "p95", "p99"]:
                summary_results[benchmark][algo][metric].append(
                    stats[algo][metric]
                )
        

# ------------------ FINAL SUMMARY PLOTS ------------------

metrics = ["avg", "p75", "p90", "p95", "p99"]

metric_titles = {
    "avg": "Average FCT",
    "p75": "75th Percentile FCT",
    "p90": "90th Percentile FCT",
    "p95": "95th Percentile FCT",
    "p99": "99th Percentile FCT"
}

marker_map = {
    "conga": "o",
    #"conga_new": "x",
    #"conga-ecmp": "X",
    "ecmp": "s",
    #"ecmp_new": "*",
    
    "lrtt_true": "1",
    "lrtt_false": "2",
    
    "weighted": "D",
    "random": "v",
    "top2": "P",
    
    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X",
}

color_map = {
    "conga": "tab:blue",
    #"conga_new": "tab:orange",
    #"conga-ecmp": "tab:green",
    "ecmp": "tab:red",
    #"ecmp_new": "tab:purple",
    
    "lrtt_true": "tab:cyan",
    "lrtt_false": "tab:cyan",
    
    "weighted": "tab:pink",
    "random": "tab:gray",
    "top2": "tab:olive",
    
    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
    
}

label_map = {
    "conga": "Conga",
    "conga_new": "Conga_New",
    "conga-ecmp": "Conga-ECMP",
    "ecmp": "ECMP",
    #"ecmp_new": "ECMP_New",
    
    "lrtt_true": "Lowest RTT (Timeout)",
    "lrtt_false": "Lowest RTT (No Timeout)",
    
    "weighted": "Weighted ECMP",
    "random": "Power-of-2 Random",
    "top2": "Power-of-2 Top2",
    
    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)",
}


# ------------------ GLOBAL Y-LIMITS ------------------

global_limits = {}

for metric in metrics:

    ymin = float("inf")
    ymax = float("-inf")

    for benchmark in summary_results:
        for algo in summary_results[benchmark]:
            if algo == "loads":
                continue

            values = summary_results[benchmark][algo][metric]

            ymin = min(ymin, min(values))
            ymax = max(ymax, max(values))

    # Add 5% padding
    margin = 0.05 * (ymax - ymin) if ymax > ymin else 1

    global_limits[metric] = (
        ymin - margin,
        ymax + margin
    )

for benchmark in summary_results:

    loads_x = summary_results[benchmark]["loads"]

    for metric in metrics:

        plt.figure(figsize=(8, 5))

        for algo in [k for k in summary_results[benchmark] if k != "loads"]:

            linestyle = "--" if algo.endswith("_false") else "-"

            plt.plot(
                loads_x,
                summary_results[benchmark][algo][metric],
                marker=marker_map.get(algo, "o"),
                color=color_map.get(algo, None),
                linewidth=2,
                label=label_map.get(algo, algo.upper())
            )

        plt.xlabel("Load")
        plt.ylabel(metric_titles[metric])
        plt.title(f"{benchmark} {metric_titles[metric]} vs Load")

        # plt.ylim(global_limits[metric])

        plt.grid(True)
        plt.legend()

        plt.tight_layout()

        plt.savefig(
            f"{save_dir}/{benchmark}_{metric}_vs_load.png",
            dpi=300
        )

        plt.close()

print("\nDONE: All comparison plots generated!")


=== private_enterprise | Load 0.1 ===
conga Avg: 0.0064150025 P75: 0.00475125 P90: 0.01875840000000001 P95: 0.03004255 P99: 0.059379100000000046
ecmp Avg: 0.0064301268333333324 P75: 0.00486475 P90: 0.0186386 P95: 0.0292865 P99: 0.05902727000000005
weighted_true Avg: 0.0064425141666666665 P75: 0.0047775 P90: 0.018935400000000015 P95: 0.029426250000000015 P99: 0.05925325000000003
random_true Avg: 0.006441598833333333 P75: 0.004795499999999999 P90: 0.01872650000000001 P95: 0.029788750000000003 P99: 0.05885897000000009
lrtt_true Avg: 0.0064425141666666665 P75: 0.0047775 P90: 0.018935400000000015 P95: 0.029426250000000015 P99: 0.05925325000000003
top2_true Avg: 0.0064425141666666665 P75: 0.0047775 P90: 0.018935400000000015 P95: 0.029426250000000015 P99: 0.05925325000000003
weighted_false Avg: 0.006433049833333333 P75: 0.004878 P90: 0.018461700000000025 P95: 0.029637550000000012 P99: 0.05925316000000003
random_false Avg: 0.0064412071666666675 P75: 0.00482725 P90: 0.018532900000000005 P95: 0

KeyboardInterrupt: 

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results"
)

graph_dir = save_dir / "graph"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

algorithms = [
    "conga",
    #"conga_new",
    #"conga-ecmp",
    "ecmp",
    #"ecmp_new",
    
    "lrtt_true",
    "lrtt_false",
    
    #"weighted",
    #"random",
    #"top2"
    
    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    "top2_true",
    "top2_false"
]

label_map = {
    "conga": "Conga",
    #"conga_new": "Conga_New",
    #"conga-ecmp": "Conga-ECMP",
    "ecmp": "ECMP",
    #"ecmp_new": "ECMP_New",
    #"rtt": "RTT",

    "lrtt_true": "Lowest RTT (Timeout)",
    "lrtt_false": "Lowest RTT (No Timeout)",
    
    "weighted": "Weighted ECMP",
    "random": "Power-of-2 Random",
    "top2": "Power-of-2 Top2",
    
    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)"
}

color_map = {
    "conga": "tab:blue",
    #"conga_new": "tab:orange",
    #"conga-ecmp": "tab:green",
    "ecmp": "tab:red",
    #"ecmp_new": "tab:purple",
    #"rtt": "tab:brown",

    "lrtt_true": "tab:cyan",
    "lrtt_false": "tab:cyan",
    
    "weighted": "tab:pink",
    "random": "tab:gray",
    "top2": "tab:olive",
    
    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
}

marker_map = {
    "conga": "o",
    #"conga_new": "x",
    #"conga-ecmp": "X",
    "ecmp": "s",
    #"ecmp_new": "*",
    #"rtt": "^",

    "lrtt_true": "1",
    "lrtt_false": "2",
    
    "weighted": "D",
    "random": "v",
    "top2": "P",
    
    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X"
}

# ------------------ GLOBAL Y-LIMITS ------------------

small_global = []
large_global = []

for benchmark in benchmarks:
    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        for algo in algorithms:
            col = f"FCT(s)_{algo}"

            if not small_df.empty:
                small_global.append(small_df[col].mean())

            if not large_df.empty:
                large_global.append(large_df[col].mean())

small_min = min(small_global)
small_max = max(small_global)

large_min = min(large_global)
large_max = max(large_global)

small_margin = 0.05 * (small_max - small_min)
large_margin = 0.05 * (large_max - large_min)

small_ylim = (small_min - small_margin, small_max + small_margin)
large_ylim = (large_min - large_margin, large_max + large_margin)

# ------------------ ANALYSIS ------------------

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    small_results = {algo: [] for algo in algorithms}
    large_results = {algo: [] for algo in algorithms}

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        # ---------- Small Flows ----------

        for algo in algorithms:

            col = f"FCT(s)_{algo}"

            if not small_df.empty:
                small_results[algo].append(
                    small_df[col].mean()
                )
            else:
                small_results[algo].append(np.nan)

        # ---------- Large Flows ----------

        for algo in algorithms:

            col = f"FCT(s)_{algo}"

            if not large_df.empty:
                large_results[algo].append(
                    large_df[col].mean()
                )
            else:
                large_results[algo].append(np.nan)

        load_vals.append(load / 10)

    # ==================================================
    # SMALL FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    for algo in algorithms:

        plt.plot(
            load_vals,
            small_results[algo],
            marker=marker_map[algo],
            color=color_map[algo],
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=label_map[algo]
        )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Small Flows (<100 KB)")
    #plt.ylim(small_ylim)
    plt.grid(True)
    plt.legend()

    out_path = graph_dir / f"{benchmark}_SMALL_vs_load.png"

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")

    # ==================================================
    # LARGE FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    for algo in algorithms:

        plt.plot(
            load_vals,
            large_results[algo],
            marker=marker_map[algo],
            color=color_map[algo],
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=label_map[algo]
        )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Large Flows (>1 MB)")
    #plt.ylim(large_ylim)
    plt.grid(True)
    plt.legend()

    out_path = graph_dir / f"{benchmark}_LARGE_vs_load.png"

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")

print("\nDone: Small vs Large flow comparison for all algorithms.")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results/graph/private_enterprise_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results/graph/private_enterprise_LARGE_vs_load.png

===== social_media_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results/graph/social_media_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results/graph/social_media_cloud_LARGE_vs_load.png

===== commercial_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results/graph/commercial_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results/graph/commercial_cloud_LARGE_vs_load.png

===== university =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results/graph/univer

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_new_test_1_sym/analysis_results"
)

baseline = "ecmp"

graph_dir = save_dir / f"graph/fct/normalised_vs_{baseline}"

graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024
LARGE = 1 * 1024 * 1024

baseline = "ecmp"

algorithms = [
    "conga",
    #"conga_new",
    #"conga-ecmp",
    #"ecmp",
    #"rtt",

    "lrtt_true",
    "lrtt_false",
    #"weighted",
    #"random",
    #"top2"
    
    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    "top2_true",
    "top2_false"

]

label_map = {
    "conga": "Conga",
    #"conga_new": "Conga_New",
    #"conga-ecmp": "Conga-ECMP",
    #"ecmp": "ECMP",
    #"rtt": "RTT",

    "lrtt_true": "Lowest RTT (Timeout)",
    "lrtt_false": "Lowest RTT (No Timeout)",
    
    "weighted": "Weighted ECMP",
    "random": "Power-of-2 Random",
    "top2": "Power-of-2 Top2",
    
    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)"
}

color_map = {
    "conga": "tab:blue",
    #"conga_new": "tab:orange",
    #"conga-ecmp": "tab:green",
    #"ecmp": "tab:red",
    #"rtt": "tab:brown",

    "lrtt_true": "tab:cyan",
    "lrtt_false": "tab:cyan",
    
    "weighted": "tab:purple",
    "random": "tab:gray",
    "top2": "tab:olive",
    
    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
}

marker_map = {
    "conga": "o",
    #"conga_new": "x",
    #"conga-ecmp": "X",
    #"ecmp": "*",
    #"rtt": "s",

    "lrtt_true": "1",
    "lrtt_false": "2",
    
    "weighted": "^",
    "random": "D",
    "top2": "P",
    
    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X"
}


# ------------------ GLOBAL Y-LIMITS ------------------

small_global = []
large_global = []

for benchmark in benchmarks:
    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        df = df[df[f"FCT(s)_{baseline}"] > 0]

        if df.empty:
            continue

        for algo in algorithms:
            df[f"{algo}_vs_{baseline}"] = (
                df[f"FCT(s)_{algo}"] /
                df[f"FCT(s)_{baseline}"]
            )

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        for algo in algorithms:

            if not small_df.empty:
                small_global.append(
                    small_df[f"{algo}_vs_{baseline}"].mean()
                )

            if not large_df.empty:
                large_global.append(
                    large_df[f"{algo}_vs_{baseline}"].mean()
                )

#small_min = min(small_global)
#small_max = max(small_global)

#large_min = min(large_global)
#large_max = max(large_global)

#small_margin = 0.05 * (small_max - small_min)
#large_margin = 0.05 * (large_max - large_min)

#small_ylim = (
#    small_min - small_margin,
#    small_max + small_margin
#)

#large_ylim = (
#    large_min - large_margin,
#    large_max + large_margin
#)

# =========================================================
# MAIN
# =========================================================

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    small_results = {algo: [] for algo in algorithms}
    large_results = {algo: [] for algo in algorithms}

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # -------------------------------------------------
        # Avoid divide-by-zero
        # -------------------------------------------------

        df = df[df[f"FCT(s)_{baseline}"] > 0]

        if df.empty:
            continue

        # -------------------------------------------------
        # Normalized FCT
        # -------------------------------------------------

        for algo in algorithms:
            df[f"{algo}_vs_{baseline}"] = (
                df[f"FCT(s)_{algo}"] /
                df[f"FCT(s)_{baseline}"]
            )

        # -------------------------------------------------
        # Flow size split
        # -------------------------------------------------

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        for algo in algorithms:

            col = f"{algo}_vs_{baseline}"

            if not small_df.empty:
                small_results[algo].append(
                    small_df[col].mean()
                )
            else:
                small_results[algo].append(np.nan)

            if not large_df.empty:
                large_results[algo].append(
                    large_df[col].mean()
                )
            else:
                large_results[algo].append(np.nan)

        load_vals.append(load / 10)


        # =====================================================
        # SMALL FLOWS
        # =====================================================
        
        plt.figure(figsize=(10, 7), dpi=120)
        
        for algo in algorithms:
        
            plt.plot(
                load_vals,
                small_results[algo],
                marker=marker_map[algo],
                color=color_map[algo],
                linestyle="--" if algo.endswith("_false") else "-",
                linewidth=2,
                markersize=7,
                label=f"{label_map[algo]} / ECMP"
            )
        
        plt.axhline(
            y=1,
            linestyle="--",
            color="black"
        )
        
        plt.xlabel("Network Load")
        plt.ylabel("Normalized FCT")
        plt.title(f"{benchmark}: Small Flows (<100 KB)")
        
        # Fixed y-axis
        plt.ylim(0, 1.8)
        plt.yticks(np.arange(0, 1.81, 0.2))
        
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        plt.tight_layout()
        
        plt.savefig(
            graph_dir /
            f"{benchmark}_SMALL_relative_vs_load.png",
            dpi=300,
            bbox_inches="tight"
        )
        
        plt.close()


        
        # =====================================================
        # LARGE FLOWS
        # =====================================================
        
        plt.figure(figsize=(10, 7), dpi=120)
        
        for algo in algorithms:
        
            plt.plot(
                load_vals,
                large_results[algo],
                marker=marker_map[algo],
                color=color_map[algo],
                linestyle="--" if algo.endswith("_false") else "-",
                linewidth=2,
                markersize=7,
                label=f"{label_map[algo]} / ECMP"
            )
        
        plt.axhline(
            y=1,
            linestyle="--",
            color="black"
        )
        
        plt.xlabel("Network Load")
        plt.ylabel("Normalized FCT")
        plt.title(f"{benchmark}: Large Flows (>1 MB)")
        
        # Fixed y-axis
        plt.ylim(0, 1.4)
        plt.yticks(np.arange(0, 1.41, 0.2))
        
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        plt.tight_layout()
        
        plt.savefig(
            graph_dir /
            f"{benchmark}_LARGE_relative_vs_load.png",
            dpi=300,
            bbox_inches="tight"
        )
        
        plt.close()

    

print("\nDone: Normalized FCT graphs generated.")


===== private_enterprise =====

===== social_media_cloud =====

===== commercial_cloud =====

===== university =====

Done: Normalized FCT graphs generated.
